# Суррогатное моделирование

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Суррогатное моделирование».

## Что делает метод

Суррогатная модель (эмулятор) — быстрая статистическая модель, которая воспроизводит результат дорогой вычислительной симуляции или эксперимента. Прямой расчёт может занимать часы; обученный суррогат выдаёт приближённый ответ за доли секунды. Это делает возможными перебор вариантов, анализ чувствительности и оптимизацию, недоступные при прямом счёте.

Ключевая идея: провести дорогой расчёт лишь в небольшом числе тщательно выбранных точек, а затем обучить на них модель, которая интерполирует отклик во всей области.

В этом блокноте мы построим суррогат дорогой функции с помощью гауссовского процесса — модели, которая вместе с предсказанием даёт оценку собственной неопределённости. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы оптимизируете форму крыла самолёта. Один расчёт методом конечных элементов занимает 6 часов. Вам нужно перебрать тысячи конфигураций. Прямой перебор займёт годы.

Суррогатное моделирование решает эту задачу:

1. **Вы проводите ограниченный «план эксперимента»** — выбираете, скажем, 50 конфигураций, хорошо покрывающих область параметров (а не случайно), и запускаете дорогой расчёт только в них.

2. **На этих 50 точках обучается «суррогат»** — быстрая статистическая модель (гауссовский процесс, нейросеть, метод опорных векторов), которая для любой новой конфигурации мгновенно выдаёт приближённый ответ. Одно предсказание — микросекунды вместо 6 часов.

3. **Суррогат используется вместо дорогого расчёта** для поиска оптимума, анализа чувствительности, визуализации ландшафта параметров.

Ключевые понятия:
- **Суррогатная модель** (эмулятор) — быстрая замена дорогому расчёту или эксперименту.
- **План эксперимента** — набор точек, в которых проводятся дорогие расчёты. Хороший план равномерно покрывает область параметров. Плохой оставляет пустоты, и суррогат ненадёжен в них.
- **Неопределённость суррогата** — мера того, насколько сильно суррогат может ошибаться в данной точке. Велика там, где дорогих расчётов нет; мала — где расчёты проводились.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Роль дорогой симуляции играет известная функция двух переменных. В реальной задаче на её месте был бы расчёт методом конечных элементов или натурный эксперимент. Мы «дорого» вычислим её лишь в небольшом числе точек плана эксперимента.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)

def expensive_simulation(X):
    """Имитация дорогой симуляции: отклик от двух параметров."""
    x1, x2 = X[:, 0], X[:, 1]
    return (np.sin(3 * x1) * np.cos(2 * x2)
            + 0.5 * (x1 - 0.5) ** 2 - 0.3 * x2)

# План эксперимента: 35 точек, равномерно покрывающих область.
n_train = 35
X_train = rng.uniform(0, 1, size=(n_train, 2))
y_train = expensive_simulation(X_train)

print(f"Число дорогих расчётов (обучающих точек): {n_train}")
pd.DataFrame(np.column_stack([X_train, y_train]),
             columns=["параметр_1", "параметр_2", "отклик"]).head()

## 4. Применение метода

### Шаг 1. Обучение суррогата

Гауссовский процесс — хороший выбор для суррогата: он не только предсказывает отклик, но и возвращает **стандартное отклонение** — оценку собственной неопределённости в каждой точке.

**Ядро**: `ConstantKernel * RBF` задаёт предположение, что отклик — гладкая функция параметров (близкие входы дают близкие выходы). `WhiteKernel` с малым `noise_level` добавляет числовую стабильность (в реальных задачах — моделирует шум расчёта).

**`normalize_y=True`** — центрирует и масштабирует отклик перед подбором ядра; почти всегда полезно включать.

После обучения смотрите на подобранное ядро: `length_scale` показывает, на каком масштабе изменений параметров значительно меняется отклик. Это содержательная характеристика задачи.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

# Ядро: масштаб отклика * гладкая корреляция + поправка на шум.
kernel = (ConstantKernel(1.0) * RBF(length_scale=0.2)
          + WhiteKernel(noise_level=1e-3))
surrogate = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                     n_restarts_optimizer=4, random_state=0)
surrogate.fit(X_train, y_train)
print("Суррогатная модель обучена.")
print(f"Подобранное ядро: {surrogate.kernel_}")

### Шаг 2. Проверка точности на независимых точках

Оценим качество суррогата на 200 новых точках, которые модель не видела при обучении. Это обязательный шаг: только проверка на независимых данных говорит, насколько суррогат пригоден для замены дорогой симуляции.

- **MAE** (средняя абсолютная ошибка) — в единицах самого отклика. Если отклик меняется от −1 до 1, MAE = 0.02 означает ошибку ~2 %.
- **R2** (коэффициент детерминации) — доля дисперсии отклика, объяснённая суррогатом. Значение 1.0 — идеально; выше 0.95 — суррогат пригоден для большинства задач.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

X_test = rng.uniform(0, 1, size=(200, 2))
y_test = expensive_simulation(X_test)
y_pred = surrogate.predict(X_test)

print(f"Средняя абсолютная ошибка: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"Коэффициент детерминации R2: {r2_score(y_test, y_pred):.4f}")

### Шаг 3. Карта отклика и карта неопределённости

Построим предсказание суррогата на плотной сетке 80×80 = 6400 точек. Прямая симуляция всей сетки потребовала бы 6400 дорогих расчётов. Суррогат делает это мгновенно.

Рядом покажем **карту неопределённости**: где суррогат уверен (вблизи точек плана) и где не уверен (пустоты между точками). Тёмные области на карте неопределённости — кандидаты для следующих дорогих расчётов, если нужно улучшить суррогат.

In [ ]:
import matplotlib.pyplot as plt

g = np.linspace(0, 1, 80)
G1, G2 = np.meshgrid(g, g)
grid = np.column_stack([G1.ravel(), G2.ravel()])
mean, std = surrogate.predict(grid, return_std=True)
mean = mean.reshape(G1.shape)
std = std.reshape(G1.shape)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

c0 = axes[0].contourf(G1, G2, mean, levels=20, cmap="viridis")
axes[0].scatter(X_train[:, 0], X_train[:, 1], c="white",
                edgecolors=VIZ["node_text"], s=45,
                label="Точки дорогого расчёта")
axes[0].set_title("Предсказанный отклик суррогата")
axes[0].set_xlabel("Параметр 1")
axes[0].set_ylabel("Параметр 2")
axes[0].legend(loc="upper right")
fig.colorbar(c0, ax=axes[0], label="Отклик")

c1 = axes[1].contourf(G1, G2, std, levels=20, cmap="magma")
axes[1].scatter(X_train[:, 0], X_train[:, 1], c="white",
                edgecolors=VIZ["node_text"], s=45)
axes[1].set_title("Неопределённость суррогата")
axes[1].set_xlabel("Параметр 1")
axes[1].set_ylabel("Параметр 2")
fig.colorbar(c1, ax=axes[1], label="Стандартное отклонение")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

*Левый — карта отклика.* Цвет показывает значение отклика симуляции (предсказанное суррогатом). Белые точки — позиции дорогих расчётов, на которых обучена модель. Суррогат позволяет увидеть весь ландшафт отклика, проведя лишь 35 расчётов вместо 6400.

*Правый — карта неопределённости.* Цвет показывает стандартное отклонение прогноза суррогата: чем светлее, тем менее уверен суррогат. Вблизи точек плана неопределённость мала; в «пустых» областях — велика. Если карта неопределённости показывает большие светлые области, суррогат ненадёжен в них — нужны дополнительные расчёты.

### «Ага»-эксперимент: сколько дорогих расчётов нужно?

Ключевой вопрос в суррогатном моделировании: при каком числе дорогих расчётов суррогат становится достаточно точным? Построим кривую точности R2 на независимом тесте в зависимости от числа обучающих точек.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# Фиксированный тест
rng_test = np.random.default_rng(99)
X_test_cv = rng_test.uniform(0, 1, size=(300, 2))
y_test_cv = expensive_simulation(X_test_cv)

train_sizes = [5, 10, 15, 20, 30, 50, 75, 100]
r2_scores = []
rng_cv = np.random.default_rng(7)

for n in train_sizes:
    X_tr = rng_cv.uniform(0, 1, size=(n, 2))
    y_tr = expensive_simulation(X_tr)
    k = ConstantKernel(1.0) * RBF(length_scale=0.2) + WhiteKernel(1e-3)
    gp_cv = GaussianProcessRegressor(kernel=k, normalize_y=True,
                                     n_restarts_optimizer=2, random_state=0)
    gp_cv.fit(X_tr, y_tr)
    y_pred_cv = gp_cv.predict(X_test_cv)
    r2_scores.append(r2_score(y_test_cv, y_pred_cv))

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(train_sizes, r2_scores, marker='o', color=VIZ['series'][0])
ax.axhline(0.95, color=VIZ['series'][2], linestyle='--',
           label='порог R2 = 0.95 (суррогат пригоден)')
ax.axhline(0.99, color=VIZ['series'][1], linestyle=':',
           label='порог R2 = 0.99 (отличная точность)')
ax.set_xlabel('Число дорогих расчётов (обучающих точек)')
ax.set_ylabel('R2 на независимом тесте')
ax.set_title('Точность суррогата в зависимости от числа обучающих расчётов')
ax.set_ylim([-0.05, 1.05])
ax.legend()
fig.tight_layout()
plt.show()

print("Число точек → R2:")
for n, r in zip(train_sizes, r2_scores):
    print(f"  {n:4d} точек: R2 = {r:.4f}")

**Как читать этот график.**

Горизонтальная ось — число дорогих расчётов, затраченных на обучение суррогата. Вертикальная ось — R2 на независимых тестовых точках (чем выше, тем точнее суррогат). Пунктирные горизонтали — пороги приемлемого качества. Кривая обычно быстро растёт при малом числе точек и выходит на плато: добавление лишних расчётов перестаёт улучшать суррогат. Точка «перегиба» — оптимальный бюджет расчётов для данной задачи.

## 5. Интерпретация результата

- **R2 близко к 1** означает, что суррогат воспроизводит дорогую симуляцию с высокой точностью и пригоден для замены прямого расчёта.
- **Карта отклика** получается мгновенно: суррогат предсказал 6400 точек за доли секунды вместо 6400 дорогих расчётов — в этом практическая ценность метода.
- **Карта неопределённости** — диагностический инструмент: светлые области показывают, где добавить следующие расчёты для улучшения суррогата. Это принцип **адаптивного планирования эксперимента** (активного обучения): не тратьте расчёты там, где суррогат уже уверен.
- **Кривая «R2 vs число расчётов»** помогает найти экономически оправданный бюджет расчётов для данной задачи.
- Суррогат достоверен только внутри области, покрытой обучающими точками. Экстраполяция за её пределы опасна, и карта неопределённости это явно показывает.

## 6. Попробуйте на своих данных

Замените демонстрационную функцию результатами своей симуляции или эксперимента.

1. Подготовьте таблицу: столбцы входных параметров и столбец измеренного отклика.
2. Снимите комментарии в ячейке ниже и укажите файл с результатами расчётов.
3. Выполните ячейки раздела 4. Для входов размерности больше двух карты отклика стройте по парам параметров, зафиксировав остальные.

## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры и посмотрите на результат:

- **`n_train`** (число обучающих точек): уменьшите до 10 или увеличьте до 100. Как меняются R2, карта отклика и карта неопределённости?
- **Случайный план vs структурированный**: замените `rng.uniform(0, 1, size=(n_train, 2))` на регулярную сетку `np.mgrid[0:1:6j, 0:1:6j].reshape(2, -1).T` (36 точек). Какой план даёт лучшую точность при том же числе точек?
- **Тип ядра**: замените `RBF` на `Matern(length_scale=0.2, nu=1.5)`. Меняется ли точность?
- **Другая симуляция**: попробуйте `expensive_simulation(X) = np.sum(X, axis=1)` (линейная функция) или `np.abs(X[:,0] - 0.5) + np.abs(X[:,1] - 0.5)` (не гладкая). Для какой функции суррогату нужно больше точек?

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("simulation_results.csv")
# feature_cols = ["параметр_1", "параметр_2"]      # входы симуляции
# target_col = "отклик"                            # результат симуляции
#
# X_train = raw[feature_cols].to_numpy(dtype=float)
# y_train = raw[target_col].to_numpy(dtype=float)
#
# surrogate.fit(X_train, y_train)
# print(surrogate.kernel_)
# Далее повторите проверку точности и карты раздела 4.

## Краткие выводы

- Суррогатная модель — быстрая замена дорогому расчёту или эксперименту: обученная на немногих точках плана, она мгновенно предсказывает отклик в любой точке.
- **Гауссовский процесс** как суррогат даёт не только прогноз, но и неопределённость — ценную информацию о том, где суррогат надёжен.
- **Качество суррогата** нужно всегда проверять на независимых точках (R2, MAE), а не только на обучающих.
- **Карта неопределённости** показывает, куда ставить следующие дорогие расчёты для максимального улучшения суррогата.
- Суррогат надёжен только внутри области, покрытой обучающими расчётами. Экстраполяция за её пределы опасна.
- Кривая «R2 vs число точек» помогает найти оптимальный бюджет: обычно достаточно десятков точек, а не тысяч.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Суррогат гауссовского процесса показал R²=0.98 на тестовых точках внутри области [0,1]², однако при попытке оптимизировать параметр за пределами этого диапазона (например, x₁=1.3) оптимизатор нашёл «минимум» там. Почему это опасно и как карта неопределённости сигнализирует об этой проблеме?
2. Кривая «R² против числа расчётов» достигает R²=0.95 уже при 20 точках и почти не растёт до 100. Коллега предлагает тем не менее добавить ещё 80 точек равномерной сеткой. При каком условии дополнительные точки реально улучшат суррогат и какой подход к выбору новых точек эффективнее равномерного?
3. Вы обучили суррогат на 35 точках дорогой симуляции и нашли оптимум по суррогату. Какую обязательную процедуру необходимо провести перед тем, как принять найденный оптимум как финальный результат, и почему суррогат в точке оптимума ненадёжен без неё?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Гауссовский процесс возвращает к априорному среднему за пределами области обучающих данных, но неопределённость там резко возрастает. Оптимизатор, минимизирующий только прогнозируемое значение, может «скатиться» в область высокой неопределённости, где суррогат произвольно низок, — это ложный оптимум. Карта неопределённости показывает большие значения σ именно за пределами [0,1]², что служит явным предупреждением об ненадёжности прогноза.
2. Дополнительные точки улучшат суррогат там, где неопределённость велика: в пустотах плана или в окрестности найденного оптимума. Равномерное добавление неэффективно, если старые точки уже хорошо покрывают область. Эффективнее — активное обучение (стратегия Expected Improvement или максимизация σ): следующую точку ставят туда, где прогноз улучшения максимален.
3. Необходимо провести хотя бы одну дорогую симуляцию (или эксперимент) непосредственно в найденной оптимальной точке и сравнить результат с прогнозом суррогата. Если реальный отклик совпадает с предсказанным в пределах σ — оптимум валиден. Если нет — суррогат в этой точке недостаточно точен, и цикл активного обучения нужно продолжить.
</details>